# NB HU36

In [6]:
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# librerias para procesamiento de archivos raster (TIFF), shapefiles y dataframes
%pip install numpy
%pip install pandas
%pip install geopandas
%pip install rasterio
%pip install --upgrade "pyarrow>=16.0.0"
%pip install "affine==2.4.0"

StatementMeta(, 0bf7312a-660e-49cc-a50e-b28806ea5dba, 29, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 63.7 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Not uninstalling pyarrow at /home/trusted-service-user/cluster-env/trident_env/lib/pyt

##### HU-36 Integrar suelos, clima, variedad y telemetría de maquinaria

In [15]:
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# ETL HU-35 Y HU-36 - Consolidacion de informacion GIS (data TIFF & shapefiles persistente en ONELIKE FABRIC)
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# Objetivo:
# 1. Crear las carpetas de consolidación.
# 2. Consultar las fuentes TIFF y la conexión de capas GIS.
# 3. Procesar únicamente archivos TIFF en las fuentes raster.
# 4. Procesar capas Shapefile completas en CAPAS_GIS.
# 5. Validar integridad y CRS EPSG:21896 de cada Shapefile.
# 6. Omitir archivos o capas cuyo nombre ya exista en el destino.
# 7. Registrar los resultados de cada archivo y capa.
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

from notebookutils import fs
from rasterio.io import MemoryFile
from rasterio.windows import Window
from pyproj import CRS
from datetime import datetime, timezone
import json


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 1. CONFIGURACIÓN DE RUTAS Y FUENTES
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

BASE = "Files"
DESTINO = f"{BASE}/CONSOLIDADO"

FUENTES = {
    "TCH_TCHM": "Data_MapasProductividad",
    "SUELOS_QUIMICOS": "Geodata_Fertilizacion",
    "COSECHA_HISTORICA": "RASTER_HISTORICOS",
    "DESPOBLACION": "Rosemberg_despoblacion",
    "CAPAS_GIS": "CapasGis"
}

CRS_OBJETIVO = 21896

print("\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 1/7] Configuración de rutas y fuentes completada.\n")


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 2. CREACIÓN DE CARPETAS
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

ejecucion = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
TEMPORAL = f"{DESTINO}/_TEMP_ETL/{ejecucion}"
resultados = []

# Crear la carpeta raíz de consolidación.
fs.mkdirs(DESTINO)

# Crear una carpeta independiente para cada fuente.
for nombre_fuente in FUENTES:
    fs.mkdirs(f"{DESTINO}/{nombre_fuente}")

# Crear la carpeta temporal de esta ejecución.
fs.mkdirs(TEMPORAL)

print("\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 2/7] Carpetas de consolidación y temporales preparadas.\n")
print(f"             Destino: {DESTINO}")
print(f"             Ejecución: {ejecucion}")


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 3. FUNCIONES DE VALIDACIÓN
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

def validar_tiff(ruta):
    """
    Comprueba disponibilidad, tamaño, apertura raster y lectura
    de una pequeña ventana de píxeles.
    Devuelve el tamaño en bytes si supera las comprobaciones.
    """

    # Leer el contenido binario del archivo.
    registro = (
        spark.read.format("binaryFile")
        .load(ruta)
        .select("length", "content")
        .first()
    )

    # Verificar que el archivo esté disponible.
    if registro is None or registro["content"] is None:
        raise ValueError("Archivo no disponible o sin contenido")

    contenido = bytes(registro["content"])
    tamano = int(registro["length"])

    # Comprobar que el archivo no esté vacío y que su tamaño coincida.
    if tamano <= 0 or len(contenido) != tamano:
        raise ValueError("Archivo vacío o tamaño inconsistente")

    # Abrir el TIFF en memoria sin modificar el archivo original.
    with MemoryFile(contenido) as memoria:
        with memoria.open() as raster:

            # Verificar dimensiones y número de bandas.
            if (
                raster.width <= 0
                or raster.height <= 0
                or raster.count < 1
            ):
                raise ValueError("Estructura raster inválida")

            # Leer una ventana pequeña para comprobar la decodificación.
            ventana = Window(
                0,
                0,
                min(10, raster.width),
                min(10, raster.height)
            )

            pixeles = raster.read(1, window=ventana)

            if pixeles.size == 0:
                raise ValueError("No se pudieron leer píxeles")

    return tamano


def leer_texto_archivo(ruta):
    """
    Lee un archivo pequeño de texto almacenado en OneLake.
    Se utiliza para interpretar el archivo de proyección .prj.
    """

    registro = (
        spark.read.format("binaryFile")
        .load(ruta)
        .select("length", "content")
        .first()
    )

    if registro is None or registro["content"] is None:
        raise ValueError("No se pudo leer el archivo PRJ")

    contenido = bytes(registro["content"])

    if len(contenido) == 0:
        raise ValueError("El archivo PRJ está vacío")

    return contenido.decode("utf-8-sig", errors="replace")


def validar_crs_shapefile(ruta_prj):
    """
    Comprueba que el archivo PRJ corresponda al sistema EPSG:21896.
    Devuelve el código EPSG identificado.
    """

    contenido_prj = leer_texto_archivo(ruta_prj)

    try:
        crs = CRS.from_wkt(contenido_prj)
    except Exception as e:
        raise ValueError(f"El archivo PRJ no contiene un WKT válido: {e}")

    epsg = crs.to_epsg(min_confidence=70)

    if epsg != CRS_OBJETIVO:
        raise ValueError(
            f"CRS no permitido. Se requiere EPSG:{CRS_OBJETIVO}; "
            f"el archivo identifica EPSG:{epsg if epsg is not None else 'desconocido'}"
        )

    return epsg


def obtener_componentes_shapefile(archivos):
    """
    Agrupa los archivos de la carpeta por nombre base.
    Reconoce las extensiones sin distinguir mayúsculas/minúsculas.
    Devuelve únicamente los grupos que contienen un archivo .shp.
    """

    grupos = {}

    for archivo in archivos:
        nombre = archivo.name

        if "." not in nombre:
            continue

        nombre_base, extension = nombre.rsplit(".", 1)
        extension = "." + extension.lower()

        clave = nombre_base.casefold()

        if clave not in grupos:
            grupos[clave] = {
                "nombre_base": nombre_base,
                "archivos": {},
                "extensiones": {}
            }

        grupos[clave]["archivos"].setdefault(extension, []).append(nombre)
        grupos[clave]["extensiones"][extension] = True

    # Una capa Shapefile se identifica por la presencia de su .shp.
    return {
        clave: grupo
        for clave, grupo in grupos.items()
        if ".shp" in grupo["archivos"]
    }


def validar_componentes_shapefile(grupo):
    """
    Comprueba que existan los cuatro componentes obligatorios:
    SHP, DBF, SHX y PRJ.
    """

    obligatorios = {".shp", ".dbf", ".shx", ".prj"}
    disponibles = set(grupo["archivos"].keys())
    faltantes = obligatorios - disponibles

    if faltantes:
        raise ValueError(
            "Faltan archivos obligatorios: "
            + ", ".join(sorted(faltantes))
        )

    # No se permite que existan dos archivos con la misma extensión
    # para una misma capa, porque no sería posible identificar el conjunto.
    for extension in obligatorios:
        if len(grupo["archivos"].get(extension, [])) != 1:
            raise ValueError(
                f"Se encontró más de un archivo {extension} "
                "para el mismo nombre base"
            )

    return True


print("\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 3/7] Funciones de validación TIFF y Shapefile preparadas.\n")


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 4. CONSULTA DE LAS FUENTES
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

for nombre_fuente, carpeta_origen in FUENTES.items():

    origen = f"{BASE}/{carpeta_origen}"
    destino = f"{DESTINO}/{nombre_fuente}"

    print(f"\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 4/7] Consultando fuente: {nombre_fuente}\n")

    try:
        archivos = fs.ls(origen)

        print(f"             Elementos encontrados: {len(archivos)}")

    except Exception as e:
        resultados.append({
            "fuente": nombre_fuente,
            "archivo": None,
            "estado": "ERROR_ORIGEN",
            "detalle": str(e)[:300]
        })

        print(f"\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[ERROR] No se pudo consultar la fuente {nombre_fuente}: {e}\n")
        print(f"\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 4/7] Fuente finalizada con error: {nombre_fuente}\n")
        continue

    # Contadores individuales para mostrar el resultado de la fuente.
    copiados_fuente = 0
    omitidos_fuente = 0
    errores_fuente = 0
    tiffs_fuente = 0
    capas_fuente = 0
    rechazados_fuente = 0

    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
    # FASE 5. VALIDACIÓN, COPIA Y PUBLICACIÓN
    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
    # PROCESAMIENTO ESPECIAL DE CAPAS GIS
    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

    if nombre_fuente == "CAPAS_GIS":

        grupos_capas = obtener_componentes_shapefile(archivos)
        capas_fuente = len(grupos_capas)

        print(f"             Capas Shapefile identificadas: {capas_fuente}")

        for clave_capa, grupo in grupos_capas.items():

            nombre_base = grupo["nombre_base"]
            nombres_archivos = [
                nombre
                for extension, lista_nombres in grupo["archivos"].items()
                for nombre in lista_nombres
            ]

            print(f"\n  [CAPA GIS] {nombre_base}")

            try:

                # Validar la existencia de los cuatro componentes obligatorios.
                validar_componentes_shapefile(grupo)
                print("  [VALIDACIÓN] Componentes SHP, DBF, SHX y PRJ presentes")

                # Validar el CRS del archivo PRJ original.
                nombre_prj = grupo["archivos"][".prj"][0]
                epsg_origen = validar_crs_shapefile(f"{origen}/{nombre_prj}")

                print(f"  [VALIDACIÓN CRS] EPSG:{epsg_origen}")

                # Omitir la capa completa si alguno de sus archivos ya existe.
                archivos_existentes = [
                    nombre
                    for nombre in nombres_archivos
                    if fs.exists(f"{destino}/{nombre}")
                ]

                if archivos_existentes:
                    resultados.append({
                        "fuente": nombre_fuente,
                        "archivo": nombre_base,
                        "estado": "OMITIDO_EXISTENTE",
                        "detalle": (
                            "La capa ya tiene archivos en el destino: "
                            + ", ".join(archivos_existentes)
                        )
                    })

                    omitidos_fuente += 1
                    print(
                        f"  [OMITIDO] {nombre_base} | "
                        "Uno o más archivos ya existen en el destino"
                    )
                    continue

                # Crear una carpeta temporal independiente para esta capa.
                temporal_capa = f"{TEMPORAL}/CAPAS_GIS/{nombre_base}"
                fs.mkdirs(temporal_capa)

                # Copiar todos los archivos asociados a la capa.
                # Además de los cuatro obligatorios, se conservan los
                # archivos complementarios que tengan el mismo nombre base.
                for nombre in nombres_archivos:

                    ruta_archivo_origen = f"{origen}/{nombre}"
                    ruta_archivo_temporal = f"{temporal_capa}/{nombre}"

                    copia_ok = fs.cp(
                        ruta_archivo_origen,
                        ruta_archivo_temporal
                    )

                    if not copia_ok or not fs.exists(ruta_archivo_temporal):
                        raise IOError(
                            f"No se pudo copiar el archivo temporal: {nombre}"
                        )

                print("  [COPIA TEMPORAL] Archivos de la capa copiados")

                # Verificar que los cuatro componentes estén en temporal.
                grupo_temporal_archivos = {
                    nombre.rsplit(".", 1)[1].lower()
                    for nombre in nombres_archivos
                    if "." in nombre
                }

                obligatorios = {".shp", ".dbf", ".shx", ".prj"}
                extensiones_temporales = {
                    "." + extension
                    for extension in grupo_temporal_archivos
                }

                if not obligatorios.issubset(extensiones_temporales):
                    raise IOError(
                        "La copia temporal no contiene todos los componentes obligatorios"
                    )

                # Validar nuevamente el CRS desde la copia temporal.
                epsg_temporal = validar_crs_shapefile(
                    f"{temporal_capa}/{nombre_prj}"
                )

                if epsg_temporal != CRS_OBJETIVO:
                    raise IOError("El CRS de la copia temporal no coincide")

                print("  [VALIDACIÓN TEMPORAL] Integridad y CRS verificados")

                # Comprobar nuevamente que no exista ningún componente
                # antes de publicar la capa.
                archivos_existentes = [
                    nombre
                    for nombre in nombres_archivos
                    if fs.exists(f"{destino}/{nombre}")
                ]

                if archivos_existentes:
                    resultados.append({
                        "fuente": nombre_fuente,
                        "archivo": nombre_base,
                        "estado": "OMITIDO_EXISTENTE",
                        "detalle": "La capa apareció en el destino antes de publicar"
                    })

                    omitidos_fuente += 1
                    print(f"  [OMITIDO] {nombre_base} | Destino ya creado")
                    continue

                # Publicar todos los componentes sin permitir sobrescritura.
                publicados = []

                try:
                    for nombre in nombres_archivos:

                        publicado = fs.mv(
                            f"{temporal_capa}/{nombre}",
                            f"{destino}/{nombre}",
                            create_path=True,
                            overwrite=False
                        )

                        if not publicado or not fs.exists(f"{destino}/{nombre}"):
                            raise IOError(
                                f"No se confirmó la publicación de {nombre}"
                            )

                        publicados.append(nombre)

                except Exception:
                    # Si la publicación falla a mitad de la capa, retirar
                    # los componentes publicados para evitar una capa parcial.
                    for nombre in publicados:
                        try:
                            fs.rm(f"{destino}/{nombre}")
                        except Exception:
                            pass
                    raise

                # Validar que los cuatro componentes estén publicados.
                for extension in obligatorios:
                    nombre_componente = grupo["archivos"][extension][0]

                    if not fs.exists(f"{destino}/{nombre_componente}"):
                        raise IOError(
                            f"No se encontró el componente publicado {nombre_componente}"
                        )

                # Validación final del CRS en el destino.
                epsg_final = validar_crs_shapefile(
                    f"{destino}/{nombre_prj}"
                )

                if epsg_final != CRS_OBJETIVO:
                    raise IOError("El CRS final no corresponde a EPSG:21896")

                resultados.append({
                    "fuente": nombre_fuente,
                    "archivo": nombre_base,
                    "estado": "COPIADO_VALIDADO",
                    "crs": f"EPSG:{epsg_final}",
                    "componentes": nombres_archivos
                })

                copiados_fuente += 1
                print(
                    f"  [OK] {nombre_base} | "
                    f"Capa completa publicada y validada en EPSG:{epsg_final}"
                )

            except ValueError as e:

                detalle = str(e)

                if "CRS" in detalle or "PRJ" in detalle or "EPSG" in detalle:
                    estado = "RECHAZADO_CRS"
                else:
                    estado = "RECHAZADO_COMPONENTES"

                resultados.append({
                    "fuente": nombre_fuente,
                    "archivo": nombre_base,
                    "estado": estado,
                    "detalle": detalle
                })

                rechazados_fuente += 1
                print(f"  [RECHAZADO] {nombre_base} | {detalle}")

            except Exception as e:

                resultados.append({
                    "fuente": nombre_fuente,
                    "archivo": nombre_base,
                    "estado": "ERROR",
                    "detalle": str(e)[:300]
                })

                errores_fuente += 1
                print(f"  [ERROR] {nombre_base} | {e}")

        print(f"\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 5/7] Fuente finalizada: \n{nombre_fuente}")
        print(f"             Capas identificadas: {capas_fuente}")
        print(f"             Capas copiadas: {copiados_fuente}")
        print(f"             Capas omitidas: {omitidos_fuente}")
        print(f"             Capas rechazadas por validación: {rechazados_fuente}")
        print(f"             Errores: {errores_fuente}")

        # No ejecutar el procesamiento TIFF para la fuente CAPAS_GIS.
        continue

    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
    # PROCESAMIENTO DE FUENTES TIFF
    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

    for archivo in archivos:

        nombre = archivo.name

        # Ignorar archivos que no sean TIFF.
        if not nombre.lower().endswith((".tif", ".tiff")):
            continue

        tiffs_fuente += 1

        ruta_origen = f"{origen}/{nombre}"
        ruta_destino = f"{destino}/{nombre}"
        ruta_temporal = f"{TEMPORAL}/{nombre_fuente}__{nombre}"

        try:

            # Omitir archivos cuyo nombre ya exista en el destino.
            if fs.exists(ruta_destino):

                resultados.append({
                    "fuente": nombre_fuente,
                    "archivo": nombre,
                    "estado": "OMITIDO_EXISTENTE"
                })

                omitidos_fuente += 1
                print(f"  [OMITIDO] {nombre} | Ya existe")
                continue

            # Validar el archivo original antes de copiarlo.
            tamano_origen = validar_tiff(ruta_origen)
            print(f"  [VALIDADO ORIGEN] {nombre}")

            # Copiar el archivo a la carpeta temporal de la ejecución.
            copia_ok = fs.cp(ruta_origen, ruta_temporal)

            if not copia_ok or not fs.exists(ruta_temporal):
                raise IOError("No se completó la copia temporal")

            print(f"  [COPIA TEMPORAL] {nombre}")

            # Validar el archivo temporal.
            tamano_temporal = validar_tiff(ruta_temporal)

            if tamano_temporal != tamano_origen:
                raise IOError("El tamaño de la copia no coincide")

            print(f"  [VALIDACIÓN TEMPORAL] {nombre}")

            # Volver a comprobar el destino para evitar sobrescribirlo.
            if fs.exists(ruta_destino):

                resultados.append({
                    "fuente": nombre_fuente,
                    "archivo": nombre,
                    "estado": "OMITIDO_EXISTENTE"
                })

                omitidos_fuente += 1
                print(f"  [OMITIDO] {nombre} | Destino ya creado")
                continue

            # Publicar la copia temporal sin permitir sobrescritura.
            publicado = fs.mv(
                ruta_temporal,
                ruta_destino,
                create_path=True,
                overwrite=False
            )

            if not publicado or not fs.exists(ruta_destino):
                raise IOError("No se confirmó la publicación")

            print(f"  [PUBLICACIÓN] {nombre}")

            # Validar el archivo ya publicado.
            tamano_final = validar_tiff(ruta_destino)

            if tamano_final != tamano_origen:
                raise IOError("El tamaño final no coincide")

            # Registrar la publicación exitosa.
            resultados.append({
                "fuente": nombre_fuente,
                "archivo": nombre,
                "estado": "COPIADO_VALIDADO",
                "bytes": tamano_final
            })

            copiados_fuente += 1
            print(f"  [OK] {nombre} | Copiado y validado \n+-+-+-+-+-")

        except Exception as e:

            # Registrar el fallo de este archivo y continuar con los demás.
            resultados.append({
                "fuente": nombre_fuente,
                "archivo": nombre,
                "estado": "ERROR",
                "detalle": str(e)[:300]
            })

            errores_fuente += 1
            print(f"  [ERROR] {nombre} | {e}")

    # Mensaje de cierre de la fuente actual.
    print(f"\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 5/7] Fuente finalizada: \n{nombre_fuente}")
    print(f"             TIFF encontrados: {tiffs_fuente}")
    print(f"             Copiados: {copiados_fuente}")
    print(f"             Omitidos: {omitidos_fuente}")
    print(f"             Errores: {errores_fuente}")


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 6. REGISTRO DE CONTROL
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

ruta_control = f"{DESTINO}/CONTROL_ETL_{ejecucion}.json"

fs.put(
    ruta_control,
    json.dumps(resultados, ensure_ascii=False, indent=2),
    overwrite=False
)

print("\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 6/7] Registro de control guardado correctamente.\n")
print(f"             Archivo: {ruta_control}")


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 7. RESUMEN FINAL
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

copiados = sum(
    r["estado"] == "COPIADO_VALIDADO"
    for r in resultados
)

omitidos = sum(
    r["estado"] == "OMITIDO_EXISTENTE"
    for r in resultados
)

errores = sum(
    r["estado"].startswith("ERROR")
    for r in resultados
)

errores_origen = sum(
    r["estado"] == "ERROR_ORIGEN"
    for r in resultados
)

rechazados_validacion = sum(
    r["estado"].startswith("RECHAZADO_")
    for r in resultados
)

print("\n")
print("#" * 65)
print("ETL HU-38 | RESUMEN FINAL DE EJECUCIÓN")
print("#" * 65)
print(f"Archivos y capas evaluados: {len(resultados)}")
print(f"Copiados y validados:       {copiados}")
print(f"Omitidos por existencia:    {omitidos}")
print(f"Rechazados por validación:  {rechazados_validacion}")
print(f"Errores de archivos:        {errores}")
print(f"Errores de fuentes:         {errores_origen}")
print(f"Registro de control:        {ruta_control}")

if errores == 0 and errores_origen == 0 and rechazados_validacion == 0:
    print("\nESTADO GENERAL: EJECUCIÓN COMPLETADA SIN ERRORES")
else:
    print("\nESTADO GENERAL: EJECUCIÓN COMPLETADA CON INCIDENCIAS")

print("\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 7/7] Proceso ETL HU-38 finalizado.\n")


StatementMeta(, 9602e6e7-6643-4cac-8ace-c4292eff5bed, 29, Finished, Available, Finished, False)


#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 1/7] Configuración de rutas y fuentes completada.


#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 2/7] Carpetas de consolidación y temporales preparadas.

             Destino: Files/CONSOLIDADO
             Ejecución: 20261104T210548Z

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 3/7] Funciones de validación TIFF y Shapefile preparadas.


#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 4/7] Consultando fuente: TCH_TCHM

             Elementos encontrados: 14
  [OMITIDO] MP_TCHM_2016.tif | Ya existe
  [OMITIDO] MP_TCHM_2017.tif | Ya existe
  [OMITIDO] MP_TCHM_2018.tif | Ya existe
  [OMITIDO] MP_TCHM_2019.tif | Ya existe
  [OMITIDO] MP_TCHM_2020.tif | Ya existe
  [OMITIDO] MP_TCHM_2021.tif | Ya existe
  [OMITIDO] MP_TCHM_2022.tif | Ya existe
  [OMITIDO] MP_TCHM_2023.tif | Ya existe
  [OMITIDO] MP_TCHM_2024.tif | Ya existe
  [OMITIDO] MP_TCHM_2025.tif | Ya existe
  [OMITIDO] MP_TCHM_2026.tif | Ya existe
  [OMITIDO] MP_TCH_2024.tif | Ya existe
  [OMITIDO] MP_TCH_2025.tif | Y